# 01) Preprocessing — CSV -> features (handover notebook)

This notebook owns **everything from raw CSVs to clean, scaled, balanced feature matrices**. It saves its outputs to `./preprocessed/` as plain `.npy` files. The companion notebook `02_graphsage.ipynb` reads from there.

## Pipeline

```
data/train.csv + data/test.csv   (Kaggle raw input)
    v
stratified train/sage_val/policy_val/test split  (cached to ./data_splits/)
    v
BERT embeddings                     (cached to ./cache/)
    v
PCA                                 (FIT ON TRAIN ONLY — no leakage)
    v
Feature scaling (z-score)           (FIT ON TRAIN ONLY)
    v
Class balancing (SMOTE on TRAIN)
    v
.npy files in ./preprocessed/       (handover output)
```

## Outputs after running this notebook

```
./preprocessed/
    X_train_bal.npy        (post-PCA, post-scaling, post-balancing)  shape: (n_train_bal, 128)
    X_sage_val.npy         (post-PCA, post-scaling)                   shape: (n_sage_val,   128)
    X_policy_val.npy       (post-PCA, post-scaling)                   shape: (n_policy_val, 128)
    X_test.npy             (post-PCA, post-scaling)                   shape: (n_test,       128)
    y_train_bal.npy        int labels                                shape: (n_train_bal,)
    y_sage_val.npy
    y_policy_val.npy
    y_test.npy
    metadata.json          (model name, PCA dim, scaler, balancing)
```

**Split rationale (4-way):** 60% train / 10% sage_val / 10% policy_val / 20% test.
sage_val is used for SAGE early-stopping; policy_val is the RL reward signal.
Keeping them separate prevents reward hacking (see RESEARCH_HANDOVER.md §9).

## How to use

1. Make sure `./data/train.csv` and `./data/test.csv` exist.
2. Run cells top to bottom. Steps that produce expensive artifacts cache them; subsequent runs are fast.
3. The final cell verifies the saved artifacts load correctly.


## 1) Imports + global setup

In [1]:
import os
import json
import math
import random
import sys
import platform
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from tqdm.auto import tqdm

# Silence the HuggingFace tokenizers fork-warning. Cosmetic only.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# scikit-learn: stratified split, PCA, scalers.
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA

# Class balancing.
from imblearn.over_sampling import SMOTE
from imblearn.combine import SMOTETomek
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours

# PyTorch (only used to run the BERT encoder; no PyG here).
import torch
from transformers import AutoTokenizer, AutoModel

import joblib

# Device selection: Apple Silicon GPU > NVIDIA GPU > CPU.
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")
print(f"Python {sys.version.split()[0]} | {platform.platform()}")


Using device: mps
Python 3.11.15 | macOS-26.4.1-arm64-arm-64bit


## 2) Configuration

All knobs are gathered here so you don't have to hunt for them. The values below match the defaults that produced our best GraphSAGE results.

> **For the student:** if you change PCA dim, embedding model, pooling, or scaling, re-run all downstream cells (caches are keyed on these settings, so you'll get fresh files in `./cache/` and `./preprocessed/`).

In [2]:
# === Paths (relative to the project root) ===
PROJECT_ROOT  = Path(".").resolve()
RAW_DATA_DIR  = Path("./data")            # Kaggle raw CSVs go here
SPLIT_DIR     = Path("./data_splits")     # cached stratified splits
CACHE_DIR     = Path("./cache")           # embeddings, PCA, intermediate features
PREPROC_DIR   = Path("./preprocessed")    # FINAL handover artifacts

for d in [SPLIT_DIR, CACHE_DIR, PREPROC_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# === Reproducibility ===
SEED = 42
def set_global_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
set_global_seed(SEED)

# === Splits (4-way: train / sage_val / policy_val / test) ===
TEST_SIZE        = 0.20   # fraction of total reserved for test
VAL_SIZE         = 0.20   # fraction of (train+val) reserved for combined val (sage_val + policy_val)
POLICY_VAL_SPLIT = 0.50   # fraction of combined val used as policy_val (rest = sage_val)
STRATIFY_COL = "sentiment"
REGENERATE_SPLITS = True  # set False after first run to reuse cached CSVs

# === Embeddings (transformer encoder) ===
HF_MODEL_NAME = "bert-base-uncased"
MAX_LEN = 128
EMBED_BATCH_SIZE = 32
EMBED_POOLING = "mean"
USE_CACHED_EMBEDDINGS = True

# === Dimensionality reduction ===
PCA_DIM = 128
PCA_WHITEN = False

# === Feature scaling ===
FEATURE_SCALING = "zscore"  # "none" | "minmax" | "zscore"

# === Class balancing ===
BALANCE_SCOPE  = "train"
BALANCE_METHOD = "smote"
BALANCE_SAMPLING_STRATEGY = "auto"
BALANCE_RANDOM_STATE = SEED
SMOTE_K_NEIGHBORS = 5
ENN_N_NEIGHBORS   = 3

# === Fixed label set ===
LABELS = ["negative", "neutral", "positive"]
label2id = {lab: i for i, lab in enumerate(LABELS)}
id2label = {i: lab for lab, i in label2id.items()}


## 3) Load + stratified split of raw CSVs

Two-stage stratified split: first carve off TEST, then split the remainder into TRAIN and VAL. Stratification preserves the class distribution in each split — important because the dataset is imbalanced.

The splits are written to `./data_splits/{train,val,test}.csv` so subsequent runs (and the GNN notebook) load the **exact same rows**.

In [3]:
def safe_read_csv(path: Path) -> pd.DataFrame:
    for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(path, encoding=enc)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(path, encoding="utf-8", encoding_errors="replace")

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    rename_map = {}
    if "Text" in df.columns and "text" not in df.columns:
        rename_map["Text"] = "text"
    if "Sentiment" in df.columns and STRATIFY_COL not in df.columns:
        rename_map["Sentiment"] = STRATIFY_COL
    df = df.rename(columns=rename_map)
    if "text" not in df.columns or STRATIFY_COL not in df.columns:
        raise ValueError(f"Expected 'text' and '{STRATIFY_COL}' columns. Found: {list(df.columns)}")
    df[STRATIFY_COL] = df[STRATIFY_COL].astype(str).str.strip().str.lower()
    return df

def load_or_create_splits():
    train_path      = SPLIT_DIR / "train.csv"
    sage_val_path   = SPLIT_DIR / "sage_val.csv"
    policy_val_path = SPLIT_DIR / "policy_val.csv"
    test_path       = SPLIT_DIR / "test.csv"

    all_exist = all(p.exists() for p in [train_path, sage_val_path, policy_val_path, test_path])
    if REGENERATE_SPLITS or not all_exist:
        print("Creating 4-way splits (train/sage_val/policy_val/test)...")
        df_tr = standardize_columns(safe_read_csv(RAW_DATA_DIR / "train.csv"))
        df_te = standardize_columns(safe_read_csv(RAW_DATA_DIR / "test.csv"))
        df_tr["__source__"] = "train_csv"
        df_te["__source__"] = "test_csv"
        df = pd.concat([df_tr, df_te], axis=0, ignore_index=True)
        df = df[["text", STRATIFY_COL, "__source__"]].dropna()

        # Stage 1: carve off test (20% of total).
        df_trv, df_test = train_test_split(
            df, test_size=TEST_SIZE, random_state=SEED, stratify=df[STRATIFY_COL])

        # Stage 2: split remainder into train (75% of remainder = 60% total)
        #          and combined val (25% of remainder = 20% total).
        df_train, df_val_combined = train_test_split(
            df_trv, test_size=VAL_SIZE, random_state=SEED, stratify=df_trv[STRATIFY_COL])

        # Stage 3: split combined val 50/50 into sage_val and policy_val.
        df_sage_val, df_policy_val = train_test_split(
            df_val_combined, test_size=POLICY_VAL_SPLIT,
            random_state=SEED, stratify=df_val_combined[STRATIFY_COL])

        df_train.reset_index(drop=True).to_csv(train_path, index=False)
        df_sage_val.reset_index(drop=True).to_csv(sage_val_path, index=False)
        df_policy_val.reset_index(drop=True).to_csv(policy_val_path, index=False)
        df_test.reset_index(drop=True).to_csv(test_path, index=False)
    else:
        print("Loading existing splits from:", SPLIT_DIR)

    df_train      = standardize_columns(safe_read_csv(train_path))
    df_sage_val   = standardize_columns(safe_read_csv(sage_val_path))
    df_policy_val = standardize_columns(safe_read_csv(policy_val_path))
    df_test       = standardize_columns(safe_read_csv(test_path))
    print(f"Split sizes: train={len(df_train):,} | sage_val={len(df_sage_val):,} "
          f"| policy_val={len(df_policy_val):,} | test={len(df_test):,}")
    return df_train, df_sage_val, df_policy_val, df_test

df_train_raw, df_sage_val_raw, df_policy_val_raw, df_test_raw = load_or_create_splits()
display(df_train_raw.head(3))


## 4) Label encoding

In [4]:
def encode_labels(series: pd.Series) -> np.ndarray:
    y = series.astype(str).str.lower().map(label2id).values
    if np.any(pd.isna(y)):
        bad = series[pd.isna(series.astype(str).str.lower().map(label2id))].unique()
        raise ValueError(f"Found unknown labels: {bad}. Expected one of {LABELS}.")
    return y.astype(int)

y_train_raw      = encode_labels(df_train_raw[STRATIFY_COL])
y_sage_val_raw   = encode_labels(df_sage_val_raw[STRATIFY_COL])
y_policy_val_raw = encode_labels(df_policy_val_raw[STRATIFY_COL])
y_test_raw       = encode_labels(df_test_raw[STRATIFY_COL])

print("Label counts:")
for name, y in [("train", y_train_raw), ("sage_val", y_sage_val_raw),
                ("policy_val", y_policy_val_raw), ("test", y_test_raw)]:
    print(f"  {name:12s}: {dict(zip(LABELS, np.bincount(y, minlength=3)))}")


## 5) BERT embeddings

Each tweet becomes a fixed-length vector by passing it through a pretrained BERT and pooling the token outputs. The embedding model is **not fine-tuned** here — we use it as a frozen feature extractor.

**Mean vs CLS pooling:** mean pooling averages real (non-pad) token positions, weighted by the attention mask. It's usually better than the `[CLS]` token for similarity-based downstream tasks (KNN, clustering) when the encoder hasn't been fine-tuned.

Embeddings are cached. The first run takes 5-15 minutes; subsequent runs are <1s.

In [5]:
def _slug(s: str) -> str:
    return s.replace("/", "__")

def embedding_cache_path(split_name: str) -> Path:
    return CACHE_DIR / f"emb_{split_name}_{_slug(HF_MODEL_NAME)}_{EMBED_POOLING}_len{MAX_LEN}.npy"

@torch.no_grad()
def embed_texts(texts, tokenizer, model, batch_size, max_len, pooling) -> np.ndarray:
    model.eval()
    all_vecs = []
    for i in tqdm(range(0, len(texts), batch_size), desc="embedding", leave=False):
        enc = tokenizer(
            texts[i:i+batch_size],
            padding=True, truncation=True,
            max_length=max_len, return_tensors="pt"
        ).to(device)
        last = model(**enc).last_hidden_state
        if pooling == "cls":
            vec = last[:, 0, :]
        elif pooling == "mean":
            mask = enc["attention_mask"].unsqueeze(-1)
            vec = (last * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
        else:
            raise ValueError(f"Unknown pooling='{pooling}'")
        all_vecs.append(vec.detach().cpu().numpy())
    return np.vstack(all_vecs).astype(np.float32)

def compute_or_load_embeddings(df: pd.DataFrame, split_name: str) -> np.ndarray:
    path = embedding_cache_path(split_name)
    if path.exists() and USE_CACHED_EMBEDDINGS:
        print(f"[cache] embeddings: {path.name}")
        return np.load(path)
    print(f"[compute] embeddings for {split_name} ({len(df):,} rows) — first run takes a few minutes")
    tok = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
    mdl = AutoModel.from_pretrained(HF_MODEL_NAME).to(device)
    emb = embed_texts(df["text"].astype(str).tolist(), tok, mdl,
                      EMBED_BATCH_SIZE, MAX_LEN, EMBED_POOLING)
    np.save(path, emb)
    return emb

emb_train      = compute_or_load_embeddings(df_train_raw,      "train")
emb_sage_val   = compute_or_load_embeddings(df_sage_val_raw,   "sage_val")
emb_policy_val = compute_or_load_embeddings(df_policy_val_raw, "policy_val")
emb_test       = compute_or_load_embeddings(df_test_raw,       "test")
print("Embedding shapes:", emb_train.shape, emb_sage_val.shape, emb_policy_val.shape, emb_test.shape)


## 6) PCA dimensionality reduction

PCA shrinks the BERT embedding (e.g., 768-dim) to `PCA_DIM` (e.g., 128). Benefits:

- Faster KNN graph construction.
- Less noise in the feature space (PCA discards low-variance directions).
- Smaller GNN input dim -> fewer parameters.

> **Critical:** PCA is fit on the **TRAIN split only**. Fitting on the union of train+val+test would let test-distribution information leak into the projection, biasing downstream metrics upward without reflecting real predictive power.

In [6]:
def pca_cache_path() -> Path:
    return CACHE_DIR / f"pca_{_slug(HF_MODEL_NAME)}_{EMBED_POOLING}_len{MAX_LEN}_dim{PCA_DIM}_whiten{int(PCA_WHITEN)}.joblib"

pca_path = pca_cache_path()
if pca_path.exists():
    print(f"[cache] PCA: {pca_path.name}")
    pca = joblib.load(pca_path)
else:
    print("[compute] PCA (fit on TRAIN only)")
    pca = PCA(n_components=PCA_DIM, whiten=PCA_WHITEN, random_state=SEED).fit(emb_train)
    joblib.dump(pca, pca_path)

X_train      = pca.transform(emb_train).astype(np.float32)
X_sage_val   = pca.transform(emb_sage_val).astype(np.float32)
X_policy_val = pca.transform(emb_policy_val).astype(np.float32)
X_test       = pca.transform(emb_test).astype(np.float32)
print("PCA feature shapes:", X_train.shape, X_sage_val.shape, X_policy_val.shape, X_test.shape)
print("Variance retained:", f"{pca.explained_variance_ratio_.sum():.4f}")


## 7) Feature scaling

Standardize features so each dim has mean 0 / std 1. Helps GNN convergence and stability (some message-passing layers are sensitive to feature magnitudes).

Same train-only-fit principle as PCA.

In [7]:
scaler = None
if FEATURE_SCALING == "zscore":
    scaler = StandardScaler()
elif FEATURE_SCALING == "minmax":
    scaler = MinMaxScaler()
elif FEATURE_SCALING != "none":
    raise ValueError(f"Unknown FEATURE_SCALING='{FEATURE_SCALING}'")

if scaler is not None:
    scaler.fit(X_train)
    X_train      = scaler.transform(X_train).astype(np.float32)
    X_sage_val   = scaler.transform(X_sage_val).astype(np.float32)
    X_policy_val = scaler.transform(X_policy_val).astype(np.float32)
    X_test       = scaler.transform(X_test).astype(np.float32)
    print(f"Scaling applied: {FEATURE_SCALING}")
else:
    print("Scaling: none")

print("Final feature shapes:", X_train.shape, X_sage_val.shape, X_policy_val.shape, X_test.shape)


## 8) Class balancing (training split only)

`BALANCE_SCOPE="train"` rebalances **only** the training split. Val and test keep their natural distribution so evaluation is honest.

| Method | What it does |
|---|---|
| `smote` (default) | Synthesizes new minority samples by interpolating between real ones |
| `tomek` | Removes pairs of overlapping samples from different classes (cleans the boundary) |
| `enn` | Drops samples whose neighbors disagree with them |
| `smote_tomek` | SMOTE then Tomek (oversample, then clean) |
| `random_under` | Randomly drops majority samples (last resort) |

> **For the student:** for the RL extension, you may want to experiment with `BALANCE_METHOD="none"` — the RL agent might *prefer* an unbalanced graph if balancing introduces synthetic edges that confuse it. Save versions with and without balancing.

In [8]:
def get_resampler(method: str):
    method = method.lower()
    if method == "none":         return None
    if method == "random_under": return RandomUnderSampler(sampling_strategy=BALANCE_SAMPLING_STRATEGY, random_state=BALANCE_RANDOM_STATE)
    if method == "tomek":        return TomekLinks(sampling_strategy=BALANCE_SAMPLING_STRATEGY)
    if method == "enn":          return EditedNearestNeighbours(sampling_strategy=BALANCE_SAMPLING_STRATEGY, n_neighbors=ENN_N_NEIGHBORS)
    if method == "smote":        return SMOTE(sampling_strategy=BALANCE_SAMPLING_STRATEGY, random_state=BALANCE_RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)
    if method == "smote_tomek":
        smote = SMOTE(sampling_strategy=BALANCE_SAMPLING_STRATEGY, random_state=BALANCE_RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)
        return SMOTETomek(sampling_strategy=BALANCE_SAMPLING_STRATEGY, random_state=BALANCE_RANDOM_STATE, smote=smote)
    raise ValueError(f"Unknown BALANCE_METHOD='{method}'")

def apply_balancing(X_tr, y_tr, X_sv, y_sv, X_pv, y_pv, X_te, y_te):
    """Balance train only; sage_val, policy_val, test keep natural distribution."""
    scope, method = BALANCE_SCOPE.lower(), BALANCE_METHOD.lower()
    rs = get_resampler(method)
    if scope == "none" or method == "none" or rs is None:
        print("Balancing: none")
        return X_tr, y_tr, X_sv, y_sv, X_pv, y_pv, X_te, y_te
    if scope == "train":
        print(f"Balancing: scope=train, method={method}")
        Xb, yb = rs.fit_resample(X_tr, y_tr)
        return Xb.astype(np.float32), yb.astype(int), X_sv, y_sv, X_pv, y_pv, X_te, y_te
    raise ValueError(f"Unknown BALANCE_SCOPE='{BALANCE_SCOPE}' (only 'train' supported with 4-way split)")

(
    X_train_bal, y_train_bal,
    X_sage_val,  y_sage_val,
    X_policy_val, y_policy_val,
    X_test_bal,  y_test_bal,
) = apply_balancing(
    X_train, y_train_raw,
    X_sage_val, y_sage_val_raw,
    X_policy_val, y_policy_val_raw,
    X_test, y_test_raw,
)
print("Train labels after balancing:", dict(zip(LABELS, np.bincount(y_train_bal, minlength=3))))


## 9) Save preprocessed artifacts (handover output)

Persist the final feature/label arrays plus a `metadata.json` that captures the config used. The GNN notebook reads from here.

In [9]:
np.save(PREPROC_DIR / "X_train_bal.npy",  X_train_bal)
np.save(PREPROC_DIR / "y_train_bal.npy",  y_train_bal)
np.save(PREPROC_DIR / "X_sage_val.npy",   X_sage_val)
np.save(PREPROC_DIR / "y_sage_val.npy",   y_sage_val)
np.save(PREPROC_DIR / "X_policy_val.npy", X_policy_val)
np.save(PREPROC_DIR / "y_policy_val.npy", y_policy_val)
np.save(PREPROC_DIR / "X_test.npy",       X_test_bal)
np.save(PREPROC_DIR / "y_test.npy",       y_test_bal)

metadata = {
    "embedding_model": HF_MODEL_NAME,
    "max_len": MAX_LEN,
    "pooling": EMBED_POOLING,
    "pca_dim": PCA_DIM,
    "pca_whiten": PCA_WHITEN,
    "feature_scaling": FEATURE_SCALING,
    "balance_scope": BALANCE_SCOPE,
    "balance_method": BALANCE_METHOD,
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
    "policy_val_split": POLICY_VAL_SPLIT,
    "labels": LABELS,
    "shapes": {
        "X_train_bal":  list(X_train_bal.shape),
        "X_sage_val":   list(X_sage_val.shape),
        "X_policy_val": list(X_policy_val.shape),
        "X_test":       list(X_test_bal.shape),
    },
    "label_counts": {
        "train_after_balancing": {LABELS[i]: int(c) for i, c in enumerate(np.bincount(y_train_bal,  minlength=3))},
        "sage_val":              {LABELS[i]: int(c) for i, c in enumerate(np.bincount(y_sage_val,   minlength=3))},
        "policy_val":            {LABELS[i]: int(c) for i, c in enumerate(np.bincount(y_policy_val, minlength=3))},
        "test":                  {LABELS[i]: int(c) for i, c in enumerate(np.bincount(y_test_bal,   minlength=3))},
    },
}
(PREPROC_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2))

print("Saved to:", PREPROC_DIR)
for f in sorted(PREPROC_DIR.iterdir()):
    print(f"  {f.name:25s} {f.stat().st_size/1024/1024:7.2f} MB")


## 10) Sanity check — verify saved artifacts load

Load the files we just wrote and confirm shapes/labels are sensible. The GNN notebook will run identical loading code.

In [10]:
X_tr_loaded  = np.load(PREPROC_DIR / "X_train_bal.npy")
y_tr_loaded  = np.load(PREPROC_DIR / "y_train_bal.npy")
X_sv_loaded  = np.load(PREPROC_DIR / "X_sage_val.npy")
y_sv_loaded  = np.load(PREPROC_DIR / "y_sage_val.npy")
X_pv_loaded  = np.load(PREPROC_DIR / "X_policy_val.npy")
y_pv_loaded  = np.load(PREPROC_DIR / "y_policy_val.npy")
X_te_loaded  = np.load(PREPROC_DIR / "X_test.npy")
y_te_loaded  = np.load(PREPROC_DIR / "y_test.npy")

assert X_tr_loaded.shape == X_train_bal.shape
assert X_sv_loaded.shape == X_sage_val.shape
assert X_pv_loaded.shape == X_policy_val.shape
assert X_te_loaded.shape == X_test_bal.shape

for arr in [y_tr_loaded, y_sv_loaded, y_pv_loaded, y_te_loaded]:
    assert arr.dtype.kind in {"i", "u"}
    assert int(arr.min()) >= 0 and int(arr.max()) < len(LABELS)

print("All artifacts loaded cleanly.")
display(pd.DataFrame([
    {"split": "train_bal",  "n_samples": len(X_tr_loaded), "feature_dim": X_tr_loaded.shape[1]},
    {"split": "sage_val",   "n_samples": len(X_sv_loaded), "feature_dim": X_sv_loaded.shape[1]},
    {"split": "policy_val", "n_samples": len(X_pv_loaded), "feature_dim": X_pv_loaded.shape[1]},
    {"split": "test",       "n_samples": len(X_te_loaded), "feature_dim": X_te_loaded.shape[1]},
]))


---

**Done.** Move on to `02_graphsage.ipynb` — it picks up from these `.npy` files.